
# Capítulo 5: Reverse — Recuperación de Señales de Audio Secretas

📖 `Libro: Investigación en Ciencia de Datos`

**Autor: Dora María Ballesteros Larrotta**  
Universidad Militar Nueva Granada (UMNG)

---

## Material complementario del libro

Este notebook acompaña el **Capítulo 5** y contiene un ejemplo de recuperación de señales de audio secretas mediante el proceso inverso de imitación (**Reverse**).

El procedimiento permite recuperar diferentes **señales de audio secretas** a partir de una misma señal objetivo (`audio.wav`) y las **claves generadas previamente mediante Imitation**, sin necesidad de utilizar los audios sintéticos obtenidos durante el proceso de imitación.

---


In [1]:
# ============================================================
# 🔓 REVERSE: RECUPERACIÓN DE CINCO MENSAJES SECRETOS
# ============================================================
# Algoritmo original: Dora María Ballesteros
# Actualizado:  21/10/2026
#
# 🎵 Entrada:
#    audio.wav
#    key1.csv ... key5.csv
#
# 🎧 Salida:
#    raudio1.wav ... raudio5.wav
#
# 🌊 DWT: sym4, nivel 2
# 🎵 Frecuencia original, sin remuestreo
# ============================================================


In [2]:
# 📚 1. IMPORTAR LIBRERÍAS

import numpy as np
import soundfile as sf
import pywt

from pathlib import Path
from IPython.display import Audio, display, Markdown

In [3]:
# ============================================================
# ⚙️ 2. CONFIGURACIÓN
# ============================================================

from pathlib import Path

# 📁 Directorio de trabajo en Google Colab
BASE_DIR = Path("/content")

# 🎵 Audio común
AUDIO_FILE = BASE_DIR / "audio.wav"

# 🔑 Las cinco claves están directamente en /content
KEY_DIR = BASE_DIR

NUM_SECRETS = 5

# 🌊 Parámetros Wavelet
WAVELET = "sym4"
LEVEL = 2


In [4]:
# ============================================================
# 🎵 3. LEER AUDIO TARGET
# ============================================================

def load_audio(filename):
    """
    Lee un audio WAV conservando su frecuencia original.

    Aplica la misma normalización utilizada en Imitation.
    """

    signal, fs = sf.read(filename)

    if signal.ndim != 1:
        raise ValueError(
            f"❌ {filename} debe ser mono."
        )

    max_amp = np.max(np.abs(signal))

    if max_amp == 0:
        raise ValueError(
            "❌ El audio contiene únicamente silencio."
        )

    signal = signal / max_amp

    return signal, fs


audio, fs = load_audio(AUDIO_FILE)

display(Markdown("## 🎵 Audio de entrada"))

print(f"📁 Archivo: {AUDIO_FILE}")
print(f"🎧 Frecuencia: {fs} Hz")
print(f"⏱️ Duración: {len(audio)/fs:.3f} s")
print(f"📊 Muestras: {len(audio)}")

display(Audio(audio, rate=fs))

## 🎵 Audio de entrada

📁 Archivo: /content/audio.wav
🎧 Frecuencia: 44100 Hz
⏱️ Duración: 8.689 s
📊 Muestras: 383184


In [5]:
# ============================================================
# 🌊 4. DESCOMPOSICIÓN WAVELET
# ============================================================

# Se realiza una sola vez porque el audio es común
# para los cinco mensajes.

coeffs = pywt.wavedec(
    audio,
    wavelet=WAVELET,
    level=LEVEL
)

# Estructura: [A2, D2, D1]
lengths = [len(c) for c in coeffs]

# Concatenar coeficientes
cI = np.concatenate(coeffs)

print("\n🌊 DESCOMPOSICIÓN DWT")
print(f"   A2: {lengths[0]} coeficientes")
print(f"   D2: {lengths[1]} coeficientes")
print(f"   D1: {lengths[2]} coeficientes")
print(f"   Total: {len(cI)} coeficientes")



🌊 DESCOMPOSICIÓN DWT
   A2: 95801 coeficientes
   D2: 95801 coeficientes
   D1: 191595 coeficientes
   Total: 383197 coeficientes


In [6]:
# ============================================================
# 🔑 5. FUNCIÓN PARA LEER Y VALIDAR CLAVES
# ============================================================

def load_key(filename, n_coeffs):
    """
    Lee una clave CSV y verifica que sea una
    permutación válida de los índices wavelet.
    """

    key = np.loadtxt(
        filename,
        delimiter=",",
        dtype=np.int64
    )

    key = np.atleast_1d(key).flatten()

    # Verificar longitud
    if len(key) != n_coeffs:
        raise ValueError(
            f"❌ {filename}: longitud incorrecta.\n"
            f"   Esperada: {n_coeffs}\n"
            f"   Encontrada: {len(key)}"
        )

    # Verificar permutación
    if not np.array_equal(
        np.sort(key),
        np.arange(n_coeffs)
    ):
        raise ValueError(
            f"❌ {filename}: no es una permutación válida."
        )

    return key


In [7]:

# ============================================================
# 🔓 6. FUNCIÓN REVERSE
# ============================================================

def reverse_audio(coefficients, key, lengths):
    """
    Recupera un mensaje secreto utilizando una clave.

    Etapas:
       1️⃣ Aplicar permutación de coeficientes.
       2️⃣ Recuperar estructura [A2, D2, D1].
       3️⃣ Aplicar IDWT.
       4️⃣ Obtener audio recuperado.
    """

    # 🔑 Recuperar el orden de los coeficientes
    cR = coefficients[key]

    # 🌊 Separar subbandas
    l1, l2, l3 = lengths

    cA2 = cR[:l1]
    cD2 = cR[l1:l1+l2]
    cD1 = cR[l1+l2:l1+l2+l3]

    # 🔄 Reconstrucción IDWT
    recovered = pywt.waverec(
        [cA2, cD2, cD1],
        wavelet=WAVELET
    )

    return recovered



In [8]:

# ============================================================
# 🎧 7. RECUPERAR LOS CINCO MENSAJES
# ============================================================

recovered_audios = {}

for i in range(1, NUM_SECRETS + 1):

    display(Markdown(
        f"### 🔐 Mensaje secreto {i}"
    ))

    # 📂 Archivos
    key_file = f"key{i}.csv"
    output_file = f"raudio{i}.wav"

    # 🔑 Leer clave
    key = load_key(
        key_file,
        len(cI)
    )

    # 🔓 Ejecutar Reverse
    recovered = reverse_audio(
        cI,
        key,
        lengths
    )

    # Ajustar longitud al audio original
    recovered = recovered[:len(audio)]

    # 💾 Guardar WAV de 24 bits
    sf.write(
        output_file,
        recovered,
        fs,
        subtype="PCM_24"
    )

    # Guardar en memoria
    recovered_audios[i] = recovered

    print(f"🔑 Clave: {key_file}")
    print(f"📊 Índices: {len(key)}")
    print(f"💾 Archivo: {output_file}")

    # 🎧 Reproducir mensaje
    display(Audio(recovered, rate=fs))

    print("─" * 55)


### 🔐 Mensaje secreto 1

🔑 Clave: key1.csv
📊 Índices: 383197
💾 Archivo: raudio1.wav


───────────────────────────────────────────────────────


### 🔐 Mensaje secreto 2

🔑 Clave: key2.csv
📊 Índices: 383197
💾 Archivo: raudio2.wav


...

🔑 Clave: key4.csv
📊 Índices: 383197
💾 Archivo: raudio4.wav


───────────────────────────────────────────────────────


### 🔐 Mensaje secreto 5

🔑 Clave: key5.csv
📊 Índices: 383197
💾 Archivo: raudio5.wav


───────────────────────────────────────────────────────


In [9]:
# ============================================================
# ✅ 8. RESUMEN FINAL
# ============================================================

display(Markdown(
    "## 🎉 ¡Recuperación finalizada!"
))

print(f"🔓 Mensajes recuperados: {len(recovered_audios)}")
print(f"🎵 Frecuencia: {fs} Hz")
print(f"🌊 Wavelet: {WAVELET}")
print(f"📊 Nivel: {LEVEL}")


## 🎉 ¡Recuperación finalizada!

🔓 Mensajes recuperados: 5
🎵 Frecuencia: 44100 Hz
🌊 Wavelet: sym4
📊 Nivel: 2
